# Fine-Tune Llama 3.2 3B for Godot UI Generation

This notebook fine-tunes Llama 3.2 3B to generate Godot 4.6 UI specifications.

**Requirements:**
- Google Colab with T4 GPU (free tier works)
- Training data: `godot_ui_training.jsonl`

**Outputs:**
- Fine-tuned LoRA adapter
- GGUF file for Ollama

## 1. Install Dependencies

In [ ]:
# Install Unsloth (optimized training)
!pip install "unsloth[colab-new]" --quiet
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" "peft<0.14.0" "accelerate<0.33.0" "bitsandbytes<0.44.0" --quiet

## 2. Load Model

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load Llama 3.2 3B Instruct with 4-bit quantization
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = 2048,
    dtype = None,  # Auto-detect
    load_in_4bit = True,  # 4-bit quantization for memory efficiency
)

print(f"Model loaded: {model.config._name_or_path}")
print(f"Max sequence length: {model.config.max_position_embeddings}")

## 3. Configure LoRA

In [ ]:
# Add LoRA adapters for efficient fine-tuning
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # LoRA rank - higher = more parameters
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,  # Usually same as rank
    lora_dropout = 0,  # 0 for code generation tasks
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # Unsloth's optimized checkpointing
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

print("LoRA adapters added successfully!")
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable_params:,} / {total_params:,} ({100*trainable_params/total_params:.2f}%)")

## 4. Prepare Training Data

In [ ]:
# Upload your training data to Colab or load from GitHub
# Option 1: Upload file directly
from google.colab import files
import os

# Check if training data exists
if not os.path.exists("godot_ui_training.jsonl"):
    print("Please upload godot_ui_training.jsonl:")
    uploaded = files.upload()
else:
    print("Training data already exists!")

# Show first few lines
!head -3 godot_ui_training.jsonl

In [ ]:
from datasets import load_dataset

# Load the JSONL dataset
dataset = load_dataset("json", data_files="godot_ui_training.jsonl", split="train")

print(f"Dataset size: {len(dataset)} examples")
print(f"\nSample example:")
print(dataset[0])

## 5. Configure Training

In [ ]:
from trl import SFTTrainer, SFTConfig

# Training configuration
training_args = SFTConfig(
    output_dir = "godot-ui-llama32",
    
    # Batch settings
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,  # Effective batch size = 8
    
    # Training schedule
    num_train_epochs = 3,
    warmup_ratio = 0.05,
    
    # Learning rate
    learning_rate = 2e-4,
    
    # Optimization
    optim = "adamw_8bit",
    weight_decay = 0.01,
    lr_scheduler_type = "linear",
    
    # Logging
    logging_steps = 10,
    save_steps = 100,
    save_total_limit = 2,
    
    # Sequence settings
    max_length = 2048,
    packing = True,  # Pack multiple examples per sequence
    
    # Train on assistant responses only
    assistant_only_loss = True,
    
    # Precision
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    
    # Other
    seed = 42,
    report_to = "none",  # Disable wandb/mlflow
)

print("Training configuration set!")

## 6. Train!

In [ ]:
# Create trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = training_args,
)

# Start training
print("Starting training...")
trainer_stats = trainer.train()
print("\nTraining complete!")

In [ ]:
# Show training stats
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"Training samples/sec: {trainer_stats.metrics['train_samples_per_second']:.2f}")
print(f"Final loss: {trainer_stats.metrics['train_loss']:.4f}")

## 7. Test the Model

In [ ]:
# Enable inference mode
FastLanguageModel.for_inference(model)

# Test prompt
test_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Generate morphing OS apps in NDJSON format. Use Godot 4.6 UI components.<|eot_id|><|start_header_id|>user<|end_header_id|>

create a simple counter app<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

# Generate
inputs = tokenizer(test_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    **inputs,
    max_new_tokens = 512,
    temperature = 0.7,
    top_p = 0.9,
    do_sample = True,
)

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("Generated UI Spec:")
print(response.split("assistant")[-1])

## 8. Save Model

In [ ]:
# Save LoRA adapter
model.save_pretrained("godot-ui-llama32-lora")
tokenizer.save_pretrained("godot-ui-llama32-lora")
print("LoRA adapter saved to godot-ui-llama32-lora/")

In [ ]:
# Save as GGUF for Ollama
model.save_pretrained_gguf(
    "godot-ui-llama32",
    tokenizer,
    quantization_method = "q4_k_m"  # Good balance of size/quality
)
print("GGUF saved to godot-ui-llama32/unsloth.F16.gguf")

## 9. Download GGUF

In [ ]:
# Download the GGUF file
from google.colab import files
files.download("godot-ui-llama32/unsloth.Q4_K_M.gguf")

## 10. Use with Ollama

After downloading the GGUF file:

```bash
# Create Modelfile
cat > Modelfile << 'EOF'
FROM ./unsloth.Q4_K_M.gguf
SYSTEM """Generate morphing OS apps in NDJSON format.

Line 1: {"id":"app_name","layout":"vbox|hbox|grid","transition":"morph"}
Lines 2-N: {"type":"...","id":"...","text":"...","action":"...","payload":{...}}
Final line: {"__end__:true}

Available types: label, button, slider, input, container, list, tree, tabs, progress, image
Actions: sys:navigate, sys:morph, agent:chat, agent:tool"""
PARAMETER temperature 0.7
PARAMETER top_p 0.9
EOF

# Create Ollama model
ollama create godot-ui -f Modelfile

# Test
ollama run godot-ui "create a calculator app"
```